In [1]:
import pandas as pd 
import numpy as np
import matplotlib as plt

In [2]:
dataset =pd.read_csv("Social_Network_Ads.csv")
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [3]:
dataset = pd.get_dummies(dataset, drop_first = True)
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,True
1,15810944,35,20000,0,True
2,15668575,26,43000,0,False
3,15603246,27,57000,0,False
4,15804002,19,76000,0,True
...,...,...,...,...,...
395,15691863,46,41000,1,False
396,15706071,51,23000,1,True
397,15654296,50,20000,1,False
398,15755018,36,33000,0,True


In [4]:
dataset = dataset.drop("User ID", axis=1)

In [5]:
dataset["Purchased"].value_counts()

Purchased
0    257
1    143
Name: count, dtype: int64

In [6]:
dataset.columns

Index(['Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [7]:
independent = dataset[['Age', 'EstimatedSalary', 'Gender_Male']]
dependent = dataset['Purchased']

In [8]:
independent.shape

(400, 3)

In [9]:
dependent

0      0
1      0
2      0
3      0
4      0
      ..
395    1
396    1
397    1
398    0
399    1
Name: Purchased, Length: 400, dtype: int64

In [10]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(independent, dependent, test_size= 1/3, random_state =0)

In [11]:
from sklearn.ensemble import RandomForestClassifier

In [14]:
from sklearn.model_selection import GridSearchCV
param_grid =  {"criterion" : ['entropy', 'gini'],
              'max_features':['sqtr' 'log2', None], 'n_estimators': [10,100]}
grid = GridSearchCV(RandomForestClassifier(), param_grid, refit =True, verbose = 3, n_jobs = -1, scoring = 'f1_weighted')

grid.fit(x_train, y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


C:\Anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
20 fits failed out of a total of 40.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
7 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Anaconda3\Lib\site-packages\sklearn\base.py", line 1358, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Anaconda3\Lib\site-packages\sklearn\base.py", line 471, in _validate_params
    validate_parameter_constraints(
    ~~~~~~~~~~~~~

,estimator,RandomForestClassifier()
,param_grid,"{'criterion': ['entropy', 'gini'], 'max_features': ['sqtrlog2', None], 'n_estimators': [10, 100]}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [15]:
re = grid.cv_results_
grid_predictions = grid.predict(x_test)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)

from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

print(cm)
print(clf_report)

[[80  5]
 [ 6 43]]
              precision    recall  f1-score   support

           0       0.93      0.94      0.94        85
           1       0.90      0.88      0.89        49

    accuracy                           0.92       134
   macro avg       0.91      0.91      0.91       134
weighted avg       0.92      0.92      0.92       134



In [16]:
from sklearn.metrics import f1_score
f1_macro = f1_score(y_test, grid_predictions, average = 'weighted')
print(" the f1_macro value for best parameter {}:". format(grid.best_params_), f1_macro)

 the f1_macro value for best parameter {'criterion': 'gini', 'max_features': None, 'n_estimators': 100}: 0.9177273336698674


In [17]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test, grid.predict_proba(x_test)[:,1])

0.9705882352941176

In [18]:
table = pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.003137,0.001556,0.000000,0.000000,entropy,sqtrlog2,10,"{'criterion': 'entropy', 'max_features': 'sqtr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
1,0.003691,0.001091,0.000000,0.000000,entropy,sqtrlog2,100,"{'criterion': 'entropy', 'max_features': 'sqtr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
2,0.087654,0.006020,0.020539,0.004780,entropy,None,10,"{'criterion': 'entropy', 'max_features': None,...",0.847141,0.868752,0.832483,0.868632,0.923510,0.868104,0.030916,3
3,0.613773,0.068035,0.034044,0.006485,entropy,None,100,"{'criterion': 'entropy', 'max_features': None,...",0.847141,0.850809,0.833323,0.887907,0.924528,0.868742,0.033233,2
4,0.004195,0.001269,0.000000,0.000000,gini,sqtrlog2,10,"{'criterion': 'gini', 'max_features': 'sqtrlog...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
5,0.002964,0.001235,0.000000,0.000000,gini,sqtrlog2,100,"{'criterion': 'gini', 'max_features': 'sqtrlog...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
6,0.080050,0.014668,0.023116,0.003146,gini,None,10,"{'criterion': 'gini', 'max_features': None, 'n...",0.804764,0.811321,0.851527,0.851527,0.903610,0.844550,0.035419,4
7,0.595418,0.011325,0.032999,0.005766,gini,None,100,"{'criterion': 'gini', 'max_features': None, 'n...",0.826263,0.868752,0.851527,0.887907,0.924528,0.871796,0.033268,1


In [19]:
age_input = float(input("Age:"))
salary_input= float (input("EstimatedSalary:"))
sex_male_input = int(input("Gender_Male:"))

Age: 23
EstimatedSalary: 490000
Gender_Male: 1


In [20]:
future_prediction = grid.predict([[ age_input, salary_input, sex_male_input]])

C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [21]:
future_prediction

array([0])